# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Mã nguồn thực nghiệm được module hoá trong thư mục `src/` và có thể chạy trực tiếp từ notebook này hoặc qua lệnh CLI.

## 0. Môi trường & Thiết lập GPU

In [ ]:
!nvidia-smi
# Cài đặt các thư viện nếu cần thiết
# !pip install -r requirements-colab.txt

In [ ]:
# Cấu hình Kaggle nếu tải tự động trên Colab/Server qua API
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception:
    pass  # Dùng file ~/.kaggle/kaggle.json hoặc biến môi trường đã có sẵn

## 1. Dữ liệu: tải và chia split theo phiên quay (Session-based Split)

In [ ]:
# Tải dữ liệu qua Kaggle API và phân chia session
# Hoặc dùng: !python scripts/prepare_data.py --zip <đường_dẫn_zip>
!python scripts/prepare_data.py --kaggle

In [ ]:
import json
from pathlib import Path
from PIL import Image
from scripts.labcommon import read_jsonl

# Kiểm tra số lượng ảnh và box từng split
meta = json.loads(Path('data/splits/meta.json').read_text())
print('Dataset version:', meta.get('dataset_version'))
for s, info in meta.get('counts', {}).items():
    print(f"{s:5s}: {info['images']} ảnh, boxes: {info['boxes']}")

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from src.label_utils import yolo_to_box, objects_to_answer, box_to_pixel, PROMPT_NORM1000

# Xem ví dụ chuyển nhãn sang JSON grounding
train_samples = read_jsonl('data/splits/train.jsonl')[:3]
for sample in train_samples:
    ans = objects_to_answer(sample['objects'])
    print(f"Image: {sample['image_id']} -> Grounding JSON:")
    print(ans)
    print('-' * 60)

## 3. E0: Zero-shot (Đánh giá mô hình gốc chưa qua huấn luyện)

In [ ]:
# Chạy suy luận Zero-shot trên tập test và tự động chấm điểm ra report/evidence/e0.json
!python src/infer.py --config configs/e0.yaml --name E0 --split test

## 4. Dựng dữ liệu huấn luyện: Chat template và Mask loss

In [ ]:
from src.dataset import RoadDamageDataset, encode_example
from src.model_utils import load_base_model_and_processor

print('Kiểm tra dataset loader và mask loss...')
ds = RoadDamageDataset('data/splits/train.jsonl', data_root='data')
sample = ds[0]
print(f"Mẫu 0: image_id={sample['image_id']}, answer={sample['answer']}")

## 5 & 6. Huấn luyện các thí nghiệm E1, E2, E3 (QLoRA, Projector, ViT LoRA)

In [ ]:
# --- Huấn luyện E1: LoRA chỉ LLM (252 modules) ---
!python src/train.py --config configs/e1.yaml

In [ ]:
# --- Huấn luyện E2: LoRA LLM + mở full Projector float32 (~109M params) ---
!python src/train.py --config configs/e2.yaml

In [ ]:
# --- Huấn luyện E3: LoRA LLM + mở Projector float32 + LoRA ViT (348 modules) ---
!python src/train.py --config configs/e3.yaml

## 7. Suy luận và Chấm điểm E1, E2, E3

In [ ]:
# Suy luận và chấm điểm E1
!python src/infer.py --config configs/e1.yaml --name E1 --split test

In [ ]:
# Suy luận và chấm điểm E2
!python src/infer.py --config configs/e2.yaml --name E2 --split test

In [ ]:
# Suy luận và chấm điểm E3
!python src/infer.py --config configs/e3.yaml --name E3 --split test

## 8. E4: Ablation Study (Chọn biến LoRA rank: r=4 vs r=64)

In [ ]:
# Huấn luyện và chấm điểm E4 với rank=4
!python src/train.py --config configs/e1.yaml --name E4_rank4 --rank 4 --output-dir work/checkpoints/e4_rank4
!python src/infer.py --config configs/e1.yaml --name E4_rank4 --checkpoint-dir work/checkpoints/e4_rank4 --evidence-out report/evidence/e4_rank_4.json --split test

In [ ]:
# Huấn luyện và chấm điểm E4 với rank=64
!python src/train.py --config configs/e1.yaml --name E4_rank64 --rank 64 --output-dir work/checkpoints/e4_rank64
!python src/infer.py --config configs/e1.yaml --name E4_rank64 --checkpoint-dir work/checkpoints/e4_rank64 --evidence-out report/evidence/e4_rank_64.json --split test

## 9. Kiểm tra tính hợp lệ của Submission

In [ ]:
!python scripts/check_submission.py